# Jac Vulnerability Classification with CodeBERT

This notebook is formatted for **Google Colab** and loads the project files
directly from a folder. It does **not** call `files.upload()`.

Place these two ZIP files in `/content` using the Colab Files panel:

- `jac_codebert_pipeline_bundle.zip`
- `jac_vulnerability_dataset_v2_5000_bundle.zip`

Then select:

**Runtime → Change runtime type → T4 GPU**

The notebook trains two CodeBERT classifiers:

1. `SAFE` versus `VULNERABLE`
2. Ten-way vulnerability-type classification

Only normalized Jac code is used as model input. IDs, explanations, labels,
and vulnerable-line annotations are excluded from the model features.

## 1 — Configure the folder paths

In [ ]:
from pathlib import Path

# Default Colab folder. Change this one value if the files are elsewhere.
FILES_DIR = Path("/content")

# Example for Google Drive instead:
# from google.colab import drive
# drive.mount("/content/drive")
# FILES_DIR = Path("/content/drive/MyDrive/JacSec")

PIPELINE_ZIP = FILES_DIR / "jac_codebert_pipeline_bundle.zip"
DATASET_ZIP = FILES_DIR / "jac_vulnerability_dataset_v2_5000_bundle.zip"

PIPELINE_DIR = Path("/content/jac_pipeline")
DATA_DIR = Path("/content/jac_data")
OUTPUT_DIR = Path("/content/runs/jac-codebert")

print("Files folder:", FILES_DIR)
print("Expected pipeline ZIP:", PIPELINE_ZIP)
print("Expected dataset ZIP:", DATASET_ZIP)

## 2 — Verify and extract both ZIP files

In [ ]:
import shutil
import zipfile

def verify_zip_member(zip_path: Path, required_filename: str) -> str:
    if not zip_path.is_file():
        raise FileNotFoundError(
            f"Required ZIP file was not found:\n{zip_path}\n\n"
            "Place it in the folder configured by FILES_DIR."
        )

    if not zipfile.is_zipfile(zip_path):
        raise ValueError(f"Not a valid ZIP archive: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as archive:
        corrupt_member = archive.testzip()
        if corrupt_member is not None:
            raise RuntimeError(
                f"Corrupt member inside {zip_path.name}: {corrupt_member}"
            )

        members = archive.namelist()

        if required_filename in members:
            return required_filename

        matches = [
            member
            for member in members
            if Path(member).name == required_filename
        ]

        if not matches:
            raise FileNotFoundError(
                f"{required_filename!r} was not found inside {zip_path.name}.\n"
                f"Archive members:\n" + "\n".join(members)
            )

        if len(matches) > 1:
            raise RuntimeError(
                f"Multiple copies of {required_filename!r} were found:\n"
                + "\n".join(matches)
            )

        return matches[0]


pipeline_member = verify_zip_member(
    PIPELINE_ZIP,
    "jac_codebert_pipeline.py",
)
dataset_member = verify_zip_member(
    DATASET_ZIP,
    "jac_vulnerability_dataset_v2_5000.csv",
)

for directory in (PIPELINE_DIR, DATA_DIR):
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(PIPELINE_ZIP, "r") as archive:
    archive.extractall(PIPELINE_DIR)

with zipfile.ZipFile(DATASET_ZIP, "r") as archive:
    archive.extractall(DATA_DIR)

PIPELINE_FILE = PIPELINE_DIR / pipeline_member
DATASET_FILE = DATA_DIR / dataset_member
REQUIREMENTS_FILE = next(
    iter(PIPELINE_DIR.rglob("requirements-jac-codebert.txt")),
    None,
)

if not PIPELINE_FILE.is_file():
    raise FileNotFoundError(
        f"Pipeline was not extracted correctly: {PIPELINE_FILE}"
    )

if not DATASET_FILE.is_file():
    raise FileNotFoundError(
        f"Dataset was not extracted correctly: {DATASET_FILE}"
    )

print("Verification successful.")
print("Pipeline:", PIPELINE_FILE)
print("Dataset:", DATASET_FILE)
print("Dataset size:", f"{DATASET_FILE.stat().st_size / 1_000_000:.2f} MB")
print("Requirements:", REQUIREMENTS_FILE)

## 3 — Install the Python dependencies

In [ ]:
# Colab already supplies a CUDA-compatible PyTorch installation.
# Avoid reinstalling torch unless Colab reports that it is missing.
!pip install -q --upgrade         "transformers>=4.46,<5"         "datasets>=2.20,<4"         "accelerate>=0.34,<2"         safetensors         "scikit-learn>=1.4"         pandas         numpy         matplotlib

## 4 — Verify the Colab runtime and GPU

In [ ]:
import platform
import torch
import transformers
import datasets
import sklearn
import pandas as pd
import numpy as np

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("scikit-learn:", sklearn.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2,
        ),
        "GB",
    )
else:
    print(
        "WARNING: No CUDA GPU is active. "
        "Select Runtime → Change runtime type → T4 GPU."
    )

## 5 — Import the Jac CodeBERT pipeline

In [ ]:
import importlib.util
import sys

spec = importlib.util.spec_from_file_location(
    "jac_codebert_pipeline",
    PIPELINE_FILE,
)

if spec is None or spec.loader is None:
    raise ImportError(f"Could not load pipeline module: {PIPELINE_FILE}")

pipeline = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = pipeline
spec.loader.exec_module(pipeline)

print("Imported module:", pipeline.__name__)
print("Selected device:", pipeline.selected_device())

## 6 — Validate the final dataset layout and 70/30 split

In [ ]:
source_df = pd.read_csv(DATASET_FILE)

pipeline.validate_source_dataframe(source_df)

expected_columns = [
    "sample_id",
    "pair_id",
    "template_id",
    "split",
    "pair_category",
    "label",
    "code_snippet",
    "vulnerable_lines",
    "vulnerable_code",
    "explanation",
]

missing_columns = [
    column for column in expected_columns
    if column not in source_df.columns
]

if missing_columns:
    raise ValueError(
        "Dataset is missing final-layout columns: "
        + ", ".join(missing_columns)
    )

split_counts = source_df["split"].value_counts()
total_rows = len(source_df)

assert total_rows == 5000, total_rows
assert split_counts.get("train", 0) == 3500, split_counts
assert split_counts.get("validation", 0) == 1500, split_counts

print("Rows:", total_rows)
print("Vulnerable/fixed pairs:", source_df["pair_id"].nunique())
print("Source templates:", source_df["template_id"].nunique())
print("\nSplit counts:")
display(split_counts.to_frame("rows"))

print("\nLabel counts:")
display(
    source_df["label"]
    .value_counts()
    .sort_index()
    .to_frame("rows")
)

## 7 — Configure preprocessing and CodeBERT training

In [ ]:
from pathlib import Path

BASE_MODEL = "microsoft/codebert-base"

# Representation:
# "window" extracts a local semantic slice around the labeled lines.
# "full" uses the complete snippet.
REPRESENTATION = "window"
WINDOW_RADIUS = 4
SCAN_STRIDE = 2

# Code normalization.
REMOVE_COMMENTS = True
CANONICALIZE_IMPORT_ALIASES = True
STRIP_IMPORT_LINES = True
NORMALIZE_IDENTIFIERS = True
NORMALIZE_LITERALS = "semantic"

# Tokenization and training.
MAX_LENGTH = 256
EPOCHS = 4
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
EARLY_STOPPING_PATIENCE = 2
SEED = 42

# CodeBERT has 12 encoder layers.
# 0 fine-tunes all layers.
# 8 freezes embeddings and the lowest eight layers.
FREEZE_ENCODER = False
FREEZE_BOTTOM_LAYERS = 8

print(
    "Effective training batch size:",
    TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
)
print("Output directory:", OUTPUT_DIR)

## 8 — Build normalized Jac semantic windows

In [ ]:
preprocess_config = pipeline.PreprocessConfig(
    representation=REPRESENTATION,
    window_radius=WINDOW_RADIUS,
    scan_stride=SCAN_STRIDE,
    remove_comments=REMOVE_COMMENTS,
    canonicalize_import_aliases=CANONICALIZE_IMPORT_ALIASES,
    strip_import_lines=STRIP_IMPORT_LINES,
    normalize_identifiers=NORMALIZE_IDENTIFIERS,
    normalize_literals=NORMALIZE_LITERALS,
    keep_newlines=True,
    prepend_jac_token=True,
)

prepared_df = pipeline.build_window_dataframe(
    source_df,
    preprocess_config,
)

print("Prepared rows:", len(prepared_df))

print("\nBinary balance:")
display(
    prepared_df
    .groupby(["split", "binary_label"])
    .size()
    .unstack(fill_value=0)
)

print("\nVulnerability-type balance:")
display(
    prepared_df[prepared_df["label"] != "SAFE"]
    .groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
)

## 9 — Compare raw Jac source and model input

In [ ]:
example_position = 0

raw_row = source_df.iloc[example_position]
prepared_row = prepared_df.iloc[example_position]

print("LABEL")
print("=" * 80)
print(prepared_row["label"])

print("\nRAW JAC CODE")
print("=" * 80)
print(raw_row["code_snippet"])

print("\nNORMALIZED CODEBERT INPUT")
print("=" * 80)
print(prepared_row["input_text"])

## 10 — Inspect CodeBERT token lengths before training

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    use_fast=True,
    trust_remote_code=False,
)

sample_texts = prepared_df["input_text"].tolist()
token_lengths = [
    len(
        tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
    )
    for text in sample_texts
]

token_length_series = pd.Series(token_lengths)

display(
    token_length_series.describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ).to_frame("token_length")
)

truncated_count = int((token_length_series > MAX_LENGTH).sum())
print(
    f"Rows longer than MAX_LENGTH={MAX_LENGTH}: "
    f"{truncated_count} / {len(token_length_series)}"
)

## 11 — Choose smoke test or full training

    Set `SMOKE_TEST = True` for a quick one-epoch test that freezes the encoder.
    Use `False` for the real experiment.

In [ ]:
SMOKE_TEST = False

if SMOKE_TEST:
    active_epochs = 1
    active_freeze_encoder = True
    active_freeze_bottom_layers = 0
else:
    active_epochs = EPOCHS
    active_freeze_encoder = FREEZE_ENCODER
    active_freeze_bottom_layers = FREEZE_BOTTOM_LAYERS

active_configuration = {
    "epochs": active_epochs,
    "freeze_encoder": active_freeze_encoder,
    "freeze_bottom_layers": active_freeze_bottom_layers,
    "base_model": BASE_MODEL,
    "max_length": MAX_LENGTH,
}

active_configuration

## 12 — Train both CodeBERT classifiers

In [ ]:
import argparse
import shutil

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

train_args = argparse.Namespace(
    dataset=DATASET_FILE,
    output=OUTPUT_DIR,
    base_model=BASE_MODEL,
    local_files_only=False,
    representation=REPRESENTATION,
    window_radius=WINDOW_RADIUS,
    scan_stride=SCAN_STRIDE,
    keep_comments=not REMOVE_COMMENTS,
    keep_imports=not STRIP_IMPORT_LINES,
    no_canonicalize_imports=not CANONICALIZE_IMPORT_ALIASES,
    keep_identifiers=not NORMALIZE_IDENTIFIERS,
    normalize_literals=NORMALIZE_LITERALS,
    max_length=MAX_LENGTH,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    epochs=active_epochs,
    train_batch_size=TRAIN_BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_ratio=WARMUP_RATIO,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    freeze_encoder=active_freeze_encoder,
    freeze_bottom_layers=active_freeze_bottom_layers,
    seed=SEED,
)

pipeline.train_pipeline(train_args)

## 13 — Read the validation metrics

In [ ]:
import json

summary_path = OUTPUT_DIR / "summary.json"

if not summary_path.is_file():
    raise FileNotFoundError(summary_path)

training_summary = json.loads(
    summary_path.read_text(encoding="utf-8")
)

print(json.dumps(training_summary, indent=2))

print(
    "\nBinary macro F1:",
    training_summary["binary"].get("eval_macro_f1"),
)
print(
    "Type macro F1:",
    training_summary["type"].get("eval_macro_f1"),
)

## 14 — Display classification reports

In [ ]:
def load_classification_report(stage: str) -> pd.DataFrame:
    path = (
        OUTPUT_DIR
        / stage
        / "classification_report.json"
    )
    report = json.loads(path.read_text(encoding="utf-8"))
    return pd.DataFrame(report).T

print("Binary classification report")
display(load_classification_report("binary"))

print("\nVulnerability-type classification report")
display(load_classification_report("type"))

## 15 — Plot confusion matrices

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_confusion_matrix(stage: str) -> None:
    path = OUTPUT_DIR / stage / "confusion_matrix.json"
    payload = json.loads(path.read_text(encoding="utf-8"))

    labels = payload["labels"]
    matrix = np.asarray(payload["matrix"])

    figure_size = max(7, len(labels) * 0.85)
    plt.figure(figsize=(figure_size, figure_size))
    plt.imshow(matrix)
    plt.title(f"{stage.title()} confusion matrix")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.xticks(
        range(len(labels)),
        labels,
        rotation=70,
        ha="right",
    )
    plt.yticks(range(len(labels)), labels)

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            plt.text(
                column_index,
                row_index,
                str(matrix[row_index, column_index]),
                ha="center",
                va="center",
            )

    plt.tight_layout()
    plt.show()

plot_confusion_matrix("binary")
plot_confusion_matrix("type")

## 16 — Inspect incorrect and low-confidence predictions

In [ ]:
def load_predictions(stage: str) -> pd.DataFrame:
    return pd.read_csv(
        OUTPUT_DIR
        / stage
        / "validation_predictions.csv"
    )

binary_predictions = load_predictions("binary")
type_predictions = load_predictions("type")

print("Binary mistakes:", int((~binary_predictions["correct"]).sum()))
display(
    binary_predictions.loc[
        ~binary_predictions["correct"],
        [
            "sample_id",
            "pair_id",
            "true_label",
            "predicted_label",
            "confidence",
            "input_text",
        ],
    ]
    .sort_values("confidence")
    .head(20)
)

print("\nLowest-confidence binary predictions")
display(
    binary_predictions[
        [
            "sample_id",
            "true_label",
            "predicted_label",
            "confidence",
            "input_text",
        ]
    ]
    .sort_values("confidence")
    .head(20)
)

print("\nType mistakes:", int((~type_predictions["correct"]).sum()))
display(
    type_predictions.loc[
        ~type_predictions["correct"],
        [
            "sample_id",
            "pair_id",
            "true_label",
            "predicted_label",
            "confidence",
            "input_text",
        ],
    ]
    .sort_values("confidence")
    .head(20)
)

## 17 — Calculate vulnerable/fixed paired accuracy

In [ ]:
pair_predictions = binary_predictions[
    [
        "pair_id",
        "true_label",
        "predicted_label",
        "correct",
    ]
].copy()

pair_scores = (
    pair_predictions
    .groupby("pair_id")
    .agg(
        rows=("correct", "size"),
        both_correct=("correct", "all"),
    )
)

malformed_pairs = int((pair_scores["rows"] != 2).sum())
paired_accuracy = float(pair_scores["both_correct"].mean())

print("Validation pairs:", len(pair_scores))
print("Malformed validation pairs:", malformed_pairs)
print("Paired accuracy:", paired_accuracy)

## 18 — Scan an unseen Jac file from a folder

    Put a `.jac` file in `/content`, then change `JAC_FILE` below. No upload
    dialog is used.

In [ ]:
import argparse

JAC_FILE = Path("/content/example.jac")
FINDINGS_FILE = Path("/content/jac_findings.json")

if not JAC_FILE.is_file():
    print(
        "Skipping scan because this file does not exist yet:",
        JAC_FILE,
    )
else:
    scan_args = argparse.Namespace(
        model_dir=OUTPUT_DIR,
        file=JAC_FILE,
        threshold=0.60,
        max_length=MAX_LENGTH,
        batch_size=16,
        max_findings=20,
        output_json=FINDINGS_FILE,
    )

    pipeline.scan_file(scan_args)

## 19 — Display the scanner findings

In [ ]:
if FINDINGS_FILE.is_file():
    findings_payload = json.loads(
        FINDINGS_FILE.read_text(encoding="utf-8")
    )
    findings_df = pd.DataFrame(
        findings_payload.get("findings", [])
    )

    if findings_df.empty:
        print(
            "No candidate windows exceeded the configured threshold."
        )
    else:
        display(findings_df)
else:
    print("No findings file exists. Run the previous cell with a .jac file.")

## 20 — Save the complete model output to Google Drive

In [ ]:
# Optional persistence step. Colab's /content directory is temporary.

from google.colab import drive

drive.mount("/content/drive")

DRIVE_OUTPUT = Path(
    "/content/drive/MyDrive/JacSecModels/jac-codebert"
)

if DRIVE_OUTPUT.exists():
    shutil.rmtree(DRIVE_OUTPUT)

DRIVE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT)

print("Saved model output to:", DRIVE_OUTPUT)

## 21 — Create a downloadable results archive

In [ ]:
from google.colab import files

archive_path = shutil.make_archive(
    "/content/jac-codebert-results",
    "zip",
    root_dir=OUTPUT_DIR,
)

print("Created:", archive_path)
files.download(archive_path)

## Interpretation

Report:

- binary macro F1;
- vulnerability-type macro F1;
- per-class recall;
- false-positive rate;
- confusion matrices;
- vulnerable/fixed paired accuracy.

A perfect or near-perfect result on this synthetic corpus means the current
patterns are easy to distinguish. It does not prove generalization to
independently developed production Jac projects.